# CTIP AI/CV Training Notebook

This notebook keeps **only the model training workflow** for the COS30049 CTIP AI/CV prototype.

The real-time AI camera / incident detection section has been removed because the project now uses a dedicated script for that part.

This notebook trains a 2-class CLIP-based image classifier using the following dataset folder structure:

```text
project-root/
├── datasets/
│   ├── touching-plants/
│   └── touching-wildlife/
├── artifacts/
└── CTIP_CLIP_Training_Only_touching_plants_wildlife.ipynb
```

The class folder names are intentionally kept as:

- `touching-plants`
- `touching-wildlife`


## Step 0. Notebook Scope

This notebook performs the following training-only tasks:

1. Imports required libraries.
2. Defines dataset, split, and artifact paths.
3. Verifies the two required dataset folders.
4. Optionally renames images using the folder naming system.
5. Splits the dataset into training and testing sets.
6. Loads CLIP image features using `openai/clip-vit-base-patch32`.
7. Trains a small classifier head on top of frozen CLIP image embeddings.
8. Evaluates the trained model using accuracy, macro F1, classification report, and confusion matrix.
9. Saves the trained model weights and metadata for the dedicated AI camera script.


## Step 1. Imports


In [2]:
import json
import random
import shutil
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from transformers import AutoProcessor, CLIPVisionModelWithProjection


## Step 2. Paths and Configuration

The notebook expects the dataset folder to be located at:

```text
project-root/datasets/
```

Inside `datasets`, the two class folders must be named exactly:

```text
touching-plants
touching-wildlife
```


In [3]:
PROJECT_DIR = Path.cwd()
DATASET_DIR = PROJECT_DIR / "datasets"
SPLIT_DIR = DATASET_DIR / "_split_2class"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SAVE_PATH = ARTIFACT_DIR / "clip_2class_touching_species.pt"
METADATA_SAVE_PATH = ARTIFACT_DIR / "clip_2class_touching_species_metadata.json"

print("Project directory:", PROJECT_DIR.resolve())
print("Dataset directory:", DATASET_DIR.resolve())
print("Split directory:", SPLIT_DIR.resolve())
print("Artifact directory:", ARTIFACT_DIR.resolve())


Project directory: /Users/chiayuenkai/Desktop/GitHub/my-react-app-main
Dataset directory: /Users/chiayuenkai/Desktop/GitHub/my-react-app-main/datasets
Split directory: /Users/chiayuenkai/Desktop/GitHub/my-react-app-main/datasets/_split_2class
Artifact directory: /Users/chiayuenkai/Desktop/GitHub/my-react-app-main/artifacts


In [4]:
MODEL_NAME = "openai/clip-vit-base-patch32"

if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("Using device:", DEVICE)

# Folder names are also used as the training class names.
CLASS_NAMES = [
    "touching-plants",
    "touching-wildlife",
]

DISPLAY_LABELS = {
    "touching-plants": "Touching Plants",
    "touching-wildlife": "Touching Wildlife",
}

CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}

VALID_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

TRAIN_RATIO = 0.90
TEST_RATIO = 0.10
RANDOM_SEED = 42

BATCH_SIZE = 8
EPOCHS = 20
LR = 1e-3
NUM_WORKERS = 0

print("Class mapping:", CLASS_TO_IDX)


Using device: mps
Class mapping: {'touching-plants': 0, 'touching-wildlife': 1}


## Step 3. Verify Dataset Folders

This step checks that the two required folders exist and counts valid image files.


In [5]:
def get_image_files(folder: Path):
    return sorted([
        f for f in folder.iterdir()
        if f.is_file() and f.suffix.lower() in VALID_EXT
    ])


def verify_dataset():
    if not DATASET_DIR.exists():
        raise FileNotFoundError(
            f"Dataset folder not found: {DATASET_DIR.resolve()}"
            "Expected structure: datasets/touching-plants and datasets/touching-wildlife"
        )

    counts = {}
    for class_name in CLASS_NAMES:
        folder = DATASET_DIR / class_name
        if not folder.exists():
            raise FileNotFoundError(f"Missing required class folder: {folder.resolve()}")

        files = get_image_files(folder)
        counts[class_name] = len(files)
        print(f"{class_name}: {len(files)} images")

    print("Total images:", sum(counts.values()))
    return counts

counts = verify_dataset()


touching-plants: 784 images
touching-wildlife: 893 images
Total images: 1677


## Step 4. Safe Image Renaming

This renames images to follow the current folder naming system.

Example output filenames:

```text
touching_plants_0001.jpg
touching_wildlife_0001.jpg
```

The folder names remain:

```text
touching-plants
touching-wildlife
```

Set `RUN_RENAME = True` only when you want to rename the actual files.


In [6]:
RUN_RENAME = False


def rename_images_safely():
    print("=== Renaming images safely ===")
    for class_name in CLASS_NAMES:
        folder = DATASET_DIR / class_name
        files = get_image_files(folder)
        prefix = class_name.replace("-", "_")

        # First rename to temporary names to avoid filename collisions.
        temp_paths = []
        for idx, old_path in enumerate(files, start=1):
            temp_path = folder / f"__tmp_{idx:05d}{old_path.suffix.lower()}"
            if old_path != temp_path:
                old_path.rename(temp_path)
            temp_paths.append(temp_path)

        # Then rename to final clean names.
        for idx, temp_path in enumerate(sorted(temp_paths), start=1):
            final_path = folder / f"{prefix}_{idx:04d}{temp_path.suffix.lower()}"
            if temp_path != final_path:
                temp_path.rename(final_path)

        print(f"{class_name}: renamed {len(temp_paths)} images")

if RUN_RENAME:
    rename_images_safely()
    counts = verify_dataset()
else:
    print("Skipping rename. Set RUN_RENAME = True to rename files.")


Skipping rename. Set RUN_RENAME = True to rename files.


## Step 5. Split Dataset into Train and Test Sets

The dataset is split using a 90/10 train/test ratio by default.

The generated split folder will be:

```text
datasets/_split_2class/
├── train/
│   ├── touching-plants/
│   └── touching-wildlife/
└── test/
    ├── touching-plants/
    └── touching-wildlife/
```


In [7]:
def split_dataset():
    print("=== Splitting dataset ===")
    random.seed(RANDOM_SEED)

    if SPLIT_DIR.exists():
        shutil.rmtree(SPLIT_DIR)

    for split in ["train", "test"]:
        for class_name in CLASS_NAMES:
            (SPLIT_DIR / split / class_name).mkdir(parents=True, exist_ok=True)

    split_counts = {"train": Counter(), "test": Counter()}

    for class_name in CLASS_NAMES:
        src_dir = DATASET_DIR / class_name
        files = get_image_files(src_dir)
        random.shuffle(files)

        n_total = len(files)
        n_train = int(n_total * TRAIN_RATIO)
        train_files = files[:n_train]
        test_files = files[n_train:]

        for file_path in train_files:
            shutil.copy2(file_path, SPLIT_DIR / "train" / class_name / file_path.name)
            split_counts["train"][class_name] += 1

        for file_path in test_files:
            shutil.copy2(file_path, SPLIT_DIR / "test" / class_name / file_path.name)
            split_counts["test"][class_name] += 1

        print(f"{class_name}: train={len(train_files)}, test={len(test_files)}, total={n_total}")

    print("Train total:", sum(split_counts["train"].values()))
    print("Test total :", sum(split_counts["test"].values()))
    return split_counts

split_counts = split_dataset()


=== Splitting dataset ===
touching-plants: train=705, test=79, total=784
touching-wildlife: train=803, test=90, total=893
Train total: 1508
Test total : 169


## Step 6. Dataset Class


In [8]:
class ImageFolderDataset(Dataset):
    def __init__(self, root_dir: Path, processor):
        self.root_dir = Path(root_dir)
        self.processor = processor
        self.samples = []

        for class_name in CLASS_NAMES:
            class_dir = self.root_dir / class_name
            if not class_dir.exists():
                continue

            for img_path in get_image_files(class_dir):
                self.samples.append((str(img_path), CLASS_TO_IDX[class_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        pixel_values = self.processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)
        return pixel_values, torch.tensor(label, dtype=torch.long)


## Step 7. CLIP Classifier Model

The CLIP visual encoder is frozen. Only the small classifier head is trained.


In [9]:
class CLIPClassifier(nn.Module):
    def __init__(self, model_name: str, num_classes: int):
        super().__init__()
        self.clip = CLIPVisionModelWithProjection.from_pretrained(
            model_name,
            use_safetensors=True,
        )

        for param in self.clip.parameters():
            param.requires_grad = False

        embed_dim = self.clip.config.projection_dim
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, pixel_values):
        outputs = self.clip(pixel_values=pixel_values)
        image_embeds = outputs.image_embeds
        image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
        logits = self.classifier(image_embeds)
        return logits


## Step 8. Training and Evaluation Helpers


In [10]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for pixel_values, labels in tqdm(loader, leave=False):
        pixel_values = pixel_values.to(DEVICE)
        labels = labels.to(DEVICE)

        with torch.set_grad_enabled(is_training):
            logits = model(pixel_values)
            loss = criterion(logits, labels)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        preds = torch.argmax(logits, dim=1)
        total_loss += loss.item() * labels.size(0)
        all_preds.extend(preds.detach().cpu().tolist())
        all_labels.extend(labels.detach().cpu().tolist())

    avg_loss = total_loss / max(1, len(loader.dataset))
    acc = accuracy_score(all_labels, all_preds) if all_labels else 0.0
    macro_f1 = f1_score(all_labels, all_preds, average="macro") if all_labels else 0.0

    return avg_loss, acc, macro_f1, all_labels, all_preds


In [11]:
def get_class_weights(train_dataset):
    label_counts = Counter(label for _, label in train_dataset.samples)
    total = sum(label_counts.values())
    num_classes = len(CLASS_NAMES)

    weights = []
    for idx in range(num_classes):
        count = label_counts.get(idx, 1)
        weights.append(total / (num_classes * count))

    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)


## Step 9. Train Model

Run this cell to train the model and save the output to:

```text
artifacts/clip_2class_touching_species.pt
artifacts/clip_2class_touching_species_metadata.json
```


In [13]:
def train_model():
    print("=== Loading processor and datasets ===")
    processor = AutoProcessor.from_pretrained(MODEL_NAME)

    train_dataset = ImageFolderDataset(SPLIT_DIR / "train", processor)
    test_dataset = ImageFolderDataset(SPLIT_DIR / "test", processor)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )

    print(f"Train samples: {len(train_dataset)}")
    print(f"Test samples : {len(test_dataset)}")

    print("=== Building model ===")
    model = CLIPClassifier(MODEL_NAME, len(CLASS_NAMES)).to(DEVICE)

    class_weights = get_class_weights(train_dataset)
    print("Class weights:", class_weights.detach().cpu().tolist())

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LR)

    best_f1 = -1.0
    best_state = None
    history = []

    print("=== Training ===")
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc, train_f1, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        test_loss, test_acc, test_f1, y_true, y_pred = run_epoch(model, test_loader, criterion)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_macro_f1": train_f1,
            "test_loss": test_loss,
            "test_acc": test_acc,
            "test_macro_f1": test_f1,
        }
        history.append(row)

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, train_f1={train_f1:.4f} | "
            f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}, test_f1={test_f1:.4f}"
        )

        if test_f1 > best_f1:
            best_f1 = test_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    print("=== Final Evaluation ===")
    _, final_acc, final_f1, y_true, y_pred = run_epoch(model, test_loader, criterion)
    target_names = [DISPLAY_LABELS[name] for name in CLASS_NAMES]

    print("Final accuracy:", final_acc)
    print("Final macro F1:", final_f1)
    print("Classification Report:")
    print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    # Save raw model state_dict for compatibility with the dedicated AI camera script.
    torch.save(model.state_dict(), MODEL_SAVE_PATH)

    metadata = {
        "model_name": MODEL_NAME,
        "class_names": CLASS_NAMES,
        "display_labels": DISPLAY_LABELS,
        "class_to_idx": CLASS_TO_IDX,
        "idx_to_class": IDX_TO_CLASS,
        "train_ratio": TRAIN_RATIO,
        "test_ratio": TEST_RATIO,
        "random_seed": RANDOM_SEED,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LR,
        "dataset_dir": str(DATASET_DIR.resolve()),
        "split_dir": str(SPLIT_DIR.resolve()),
        "model_save_path": str(MODEL_SAVE_PATH.resolve()),
        "final_accuracy": final_acc,
        "final_macro_f1": final_f1,
        "history": history,
    }

    with open(METADATA_SAVE_PATH, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print("Saved model to:", MODEL_SAVE_PATH.resolve())
    print("Saved metadata to:", METADATA_SAVE_PATH.resolve())

    return model, processor, metadata

model, processor, metadata = train_model()


=== Loading processor and datasets ===


Train samples: 1508
Test samples : 169
=== Building model ===


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0

Class weights: [1.0695035457611084, 0.9389788508415222]
=== Training ===


...

Epoch 17/20 | train_loss=0.0439, train_acc=0.9914, train_f1=0.9914 | test_loss=0.0576, test_acc=0.9822, test_f1=0.9822


Epoch 18/20 | train_loss=0.0416, train_acc=0.9914, train_f1=0.9914 | test_loss=0.0554, test_acc=0.9822, test_f1=0.9822


Epoch 19/20 | train_loss=0.0396, train_acc=0.9920, train_f1=0.9920 | test_loss=0.0536, test_acc=0.9822, test_f1=0.9822


Epoch 20/20 | train_loss=0.0380, train_acc=0.9920, train_f1=0.9920 | test_loss=0.0518, test_acc=0.9822, test_f1=0.9822
=== Final Evaluation ===


Final accuracy: 0.9822485207100592
Final macro F1: 0.9822086535424781
Classification Report:
                   precision    recall  f1-score   support

  Touching Plants     0.9634    1.0000    0.9814        79
Touching Wildlife     1.0000    0.9667    0.9831        90

         accuracy                         0.9822       169
        macro avg     0.9817    0.9833    0.9822       169
     weighted avg     0.9829    0.9822    0.9823       169

Confusion Matrix:
[[79  0]
 [ 3 87]]
Saved model to: /Users/chiayuenkai/Desktop/GitHub/my-react-app-main/artifacts/clip_2class_touching_species.pt
Saved metadata to: /Users/chiayuenkai/Desktop/GitHub/my-react-app-main/artifacts/clip_2class_touching_species_metadata.json


## Step 10. Quick Single-Image Test

This optional cell checks one image using the trained model. It does **not** run camera detection or save incidents.


In [14]:
def predict_image(image_path: str):
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    image = Image.open(image_path).convert("RGB")
    pixel_values = processor(images=image, return_tensors="pt")["pixel_values"].to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(pixel_values)
        probs = torch.softmax(logits, dim=1).squeeze(0).detach().cpu()

    pred_idx = int(torch.argmax(probs).item())
    pred_class = IDX_TO_CLASS[pred_idx]

    result = {
        "image": str(image_path),
        "predicted_class": pred_class,
        "display_label": DISPLAY_LABELS[pred_class],
        "confidence": float(probs[pred_idx]),
        "probabilities": {
            CLASS_NAMES[i]: float(probs[i]) for i in range(len(CLASS_NAMES))
        },
    }
    return result

# Example usage after training:
# sample_image = DATASET_DIR / "touching-plants" / "touching_plants_0001.jpg"
# predict_image(sample_image)


## Notes for Integration with Dedicated AI Camera Script

This notebook only trains and saves the model.

The dedicated AI camera script should load:

```text
artifacts/clip_2class_touching_species.pt
```

The trained class order is:

```python
["touching-plants", "touching-wildlife"]
```

For display purposes, the labels can be shown as:

```python
"Touching Plants"
"Touching Wildlife"
```

No camera loop, MediaPipe hand detection, alert JSON generation, or incident evidence saving is included in this notebook.
